# Notebook pour entraîner un modèle LoRA avec Diffusers et un dataset TopDown

**Version adaptée pour environnement local avec GPU**

## Vérification de l'environnement et du GPU

In [1]:
import torch
import os
from pathlib import Path

# Vérification du GPU
print(f"PyTorch version: {torch.__version__}") 
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU détecté: {torch.cuda.get_device_name(0)}")
    print(f"Mémoire GPU disponible: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    device = "cuda"
else:
    print("GPU non disponible, utilisation du CPU")
    device = "cpu"

# Définition des chemins locaux
project_root = Path(r"C:\Users\PC\Documents\GitHub\CADOTProject")
dataset_root = project_root / "CADOT_Dataset"
train_images = dataset_root / "train"
lora_output = dataset_root / "LoRA"

print(f"\nChemins du projet:")
print(f"Racine: {project_root}")
print(f"Dataset: {dataset_root}")
print(f"Images d'entraînement: {train_images}")
print(f"Sortie LoRA: {lora_output}")

# Créer les dossiers nécessaires
lora_output.mkdir(exist_ok=True)
(train_images / "captions").mkdir(exist_ok=True)

PyTorch version: 2.9.0+cpu
CUDA disponible: False
GPU non disponible, utilisation du CPU

Chemins du projet:
Racine: C:\Users\PC\Documents\GitHub\CADOTProject
Dataset: C:\Users\PC\Documents\GitHub\CADOTProject\CADOT_Dataset
Images d'entraînement: C:\Users\PC\Documents\GitHub\CADOTProject\CADOT_Dataset\train
Sortie LoRA: C:\Users\PC\Documents\GitHub\CADOTProject\CADOT_Dataset\LoRA


## Préparer les datasets pour l'entraînement

In [4]:
import json
from collections import defaultdict, Counter

# Utiliser les chemins locaux
coco_json_path = train_images / "_annotations.coco.json"
images_dir = train_images
captions_dir = train_images / "captions"

print(f"Lecture du fichier COCO: {coco_json_path}")

# Vérifier si le fichier existe
if not coco_json_path.exists():
    print(f"ERREUR: Fichier COCO introuvable: {coco_json_path}")
    print(f"Fichiers disponibles dans {train_images}:")
    for f in train_images.glob("*"):
        print(f"  {f.name}")
else:
    with open(coco_json_path, "r") as f:
        coco = json.load(f)
    
    print(f"Dataset chargé: {len(coco['images'])} images, {len(coco['annotations'])} annotations")

    images = {img["id"]: img for img in coco["images"]}
    categories = {cat["id"]: cat["name"] for cat in coco["categories"]}
    
    print(f"Catégories disponibles: {list(categories.values())}")

    anns_per_image = defaultdict(list)
    for ann in coco["annotations"]:
        anns_per_image[ann["image_id"]].append(ann)

    def build_caption(anns):
        if not anns:
            return "top-down aerial RGB image of an urban area"
        cls_counts = Counter(categories[a["category_id"]] for a in anns)
        parts = []
        for cls, n in cls_counts.items():
            parts.append(f"{n} {cls}s" if n > 1 else f"one {cls}")
        return "top-down aerial RGB image of an urban area with " + ", ".join(parts)

    # Générer les captions
    caption_count = 0
    for img_id, img_info in images.items():
        file_name = img_info["file_name"]
        anns = anns_per_image.get(img_id, [])
        caption = build_caption(anns)
        txt_path = captions_dir / (Path(file_name).stem + ".txt")
        with txt_path.open("w") as f:
            f.write(caption)
        caption_count += 1
    
    print(f"Captions générées: {caption_count}")

Lecture du fichier COCO: C:\Users\PC\Documents\GitHub\CADOTProject\CADOT_Dataset\train\_annotations.coco.json
Dataset chargé: 3234 images, 75198 annotations
Catégories disponibles: ['small-object', 'basketball field', 'building', 'crosswalk', 'football field', 'graveyard', 'large vehicle', 'medium vehicle', 'playground', 'roundabout', 'ship', 'small vehicle', 'swimming pool', 'tennis court', 'train']
Dataset chargé: 3234 images, 75198 annotations
Catégories disponibles: ['small-object', 'basketball field', 'building', 'crosswalk', 'football field', 'graveyard', 'large vehicle', 'medium vehicle', 'playground', 'roundabout', 'ship', 'small vehicle', 'swimming pool', 'tennis court', 'train']
Captions générées: 3234
Captions générées: 3234


## Charger le modèle TopDown (ou utiliser Stable Diffusion de base)

**Note**: Si vous n'avez pas le fichier topdown.safetensors, le code utilisera Stable Diffusion 1.5 de base.

In [8]:
from diffusers import StableDiffusionPipeline
import torch

# Chemin vers le modèle TopDown (ajustez si nécessaire)
topdown_model_path = project_root / "topdown.safetensors"

try:
    if topdown_model_path.exists():
        print(f"Chargement du modèle TopDown depuis: {topdown_model_path}")
        pipe = StableDiffusionPipeline.from_single_file(
            str(topdown_model_path),
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
            use_safetensors=True
        ).to(device)
    else:
        print("Modèle TopDown non trouvé, utilisation de Stable Diffusion 1.5 de base")
        pipe = StableDiffusionPipeline.from_pretrained(
            "runwayml/stable-diffusion-v1-5",
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
            use_safetensors=True
        ).to(device)

    # Optimisations mémoire pour les nouvelles versions
    if device == "cuda":
        try:
            # Nouvelle méthode pour diffusers >= 0.30
            pipe.enable_model_cpu_offload()
            pipe.enable_attention_slicing()
            print("Optimisations mémoire activées")
        except Exception as opt_error:
            print(f"Optimisations partielles: {opt_error}")
    
    print(f"Pipeline chargé avec succès sur {device}")

except Exception as e:
    print(f"Erreur lors du chargement du modèle: {e}")
    print("Tentative avec des paramètres plus conservateurs...")
    pipe = StableDiffusionPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        torch_dtype=torch.float32
    ).to(device)

RuntimeError: Failed to import diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion because of the following error (look up to see its traceback):
cannot import name 'CLIPImageProcessor' from 'transformers' (c:\Users\PC\Documents\GitHub\CADOTProject\.venv\Lib\site-packages\transformers\__init__.py)

## Dataset personnalisé pour l'entraînement

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import random
from pathlib import Path

class CadotLoraDataset(Dataset):
    def __init__(self, img_dir, cap_dir, size=512):
        self.img_dir = Path(img_dir)
        self.cap_dir = Path(cap_dir)
        self.size = size
        
        # Chercher les images avec extensions communes
        extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp']
        self.items = []
        for ext in extensions:
            self.items.extend(list(self.img_dir.glob(ext)))
        
        self.items = sorted(self.items)
        print(f"Dataset initialisé avec {len(self.items)} images")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        img_path = self.items[idx]
        txt_path = self.cap_dir / (img_path.stem + ".txt")

        try:
            # Charger et redimensionner l'image
            image = Image.open(img_path).convert("RGB")
            image = image.resize((self.size, self.size), Image.Resampling.LANCZOS)
            
            # Charger la caption
            if txt_path.exists():
                with txt_path.open('r', encoding='utf-8') as f:
                    caption = f.read().strip()
            else:
                caption = "top-down aerial RGB image of an urban area"

            return {"pixel_values": image, "caption": caption}
        
        except Exception as e:
            print(f"Erreur lors du chargement de {img_path}: {e}")
            # Retourner une image par défaut
            image = Image.new('RGB', (self.size, self.size), color='black')
            return {"pixel_values": image, "caption": "top-down aerial RGB image of an urban area"}

# Créer le dataset
dataset = CadotLoraDataset(train_images, captions_dir)
print(f"Dataset créé avec {len(dataset)} échantillons")

## Configuration des modules LoRA

In [ ]:
# Adaptation pour les nouvelles versions de diffusers
try:
    from diffusers.models.attention_processor import LoRAAttnProcessor
except ImportError:
    # Pour les versions plus récentes
    from diffusers.models.attention_processor import LoRAAttnProcessor2_0 as LoRAAttnProcessor

unet = pipe.unet
rank = 8  # Rang LoRA (8, 16, ou 32)

print("Configuration des processeurs d'attention LoRA...")

# Collecter tous les paramètres LoRA
lora_params = []

# Appliquer LoRA aux couches d'attention - compatible avec nouvelles versions
attention_procs = {}
for name in unet.attn_processors.keys():
    cross_attention_dim = None if name.endswith("attn1.processor") else unet.config.cross_attention_dim
    if name.startswith("mid_block"):
        hidden_size = unet.config.block_out_channels[-1]
    elif name.startswith("up_blocks"):
        block_id = int(name[len("up_blocks.")])
        hidden_size = list(reversed(unet.config.block_out_channels))[block_id]
    elif name.startswith("down_blocks"):
        block_id = int(name[len("down_blocks.")])
        hidden_size = unet.config.block_out_channels[block_id]
    else:
        hidden_size = unet.config.block_out_channels[0]

    lora_attn_proc = LoRAAttnProcessor(
        hidden_size=hidden_size,
        cross_attention_dim=cross_attention_dim,
        rank=rank
    )
    attention_procs[name] = lora_attn_proc
    
    # Collecter les paramètres LoRA
    lora_params.extend(list(lora_attn_proc.parameters()))

# Appliquer tous les processeurs d'attention
unet.set_attn_processor(attention_procs)

print(f"LoRA configuré avec {len(lora_params)} paramètres à entraîner")

# Vérifier le nombre total de paramètres
total_params = sum(p.numel() for p in lora_params)
print(f"Nombre total de paramètres LoRA: {total_params:,}")

## Boucle d'entraînement optimisée pour GPU local

In [ ]:
from torchvision import transforms
from torch.utils.data import DataLoader
import torch
from tqdm.auto import tqdm
import gc

# Configuration des transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

def collate_fn(batch):
    images = [transform(item["pixel_values"]) for item in batch]
    captions = [item["caption"] for item in batch]
    images = torch.stack(images)
    return {"pixel_values": images, "captions": captions}

# Configuration de l'entraînement
batch_size = 1 if device == "cuda" else 2  # Réduire pour GPU avec moins de mémoire
num_epochs = 3
learning_rate = 1e-4

train_loader = DataLoader(
    dataset, 
    batch_size=batch_size, 
    shuffle=True, 
    collate_fn=collate_fn,
    num_workers=0  # Pas de multiprocessing sur Windows
)

# Optimiseur uniquement sur les paramètres LoRA
optimizer = torch.optim.AdamW(lora_params, lr=learning_rate)

# Composants du pipeline
vae = pipe.vae
tokenizer = pipe.tokenizer
text_encoder = pipe.text_encoder
noise_scheduler = pipe.scheduler

print(f"Début de l'entraînement:")
print(f"- Epochs: {num_epochs}")
print(f"- Batch size: {batch_size}")
print(f"- Learning rate: {learning_rate}")
print(f"- Device: {device}")

# Mettre les modèles en mode évaluation (sauf UNet)
vae.eval()
text_encoder.eval()

for epoch in range(num_epochs):
    unet.train()
    total_loss = 0
    num_batches = 0
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    
    for batch_idx, batch in enumerate(progress_bar):
        try:
            with torch.no_grad():
                # Encoder les images
                pixel_values = batch["pixel_values"].to(device)
                if device == "cuda":
                    pixel_values = pixel_values.half()
                
                latents = vae.encode(pixel_values).latent_dist.sample()
                latents = latents * vae.config.scaling_factor

                # Encoder le texte
                text_inputs = tokenizer(
                    batch["captions"],
                    padding="max_length",
                    max_length=tokenizer.model_max_length,
                    truncation=True,
                    return_tensors="pt",
                )
                encoder_hidden_states = text_encoder(
                    text_inputs.input_ids.to(device)
                )[0]

            # Ajouter du bruit
            noise = torch.randn_like(latents)
            timesteps = torch.randint(
                0, noise_scheduler.config.num_train_timesteps, 
                (latents.shape[0],), device=device
            ).long()
            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

            # Prédiction du modèle
            model_pred = unet(
                noisy_latents, 
                timesteps, 
                encoder_hidden_states=encoder_hidden_states
            ).sample

            # Calcul de la perte
            loss = torch.nn.functional.mse_loss(
                model_pred.float(), 
                noise.float()
            )

            # Rétropropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            num_batches += 1
            
            # Mise à jour de la barre de progression
            progress_bar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'avg_loss': f'{total_loss/num_batches:.4f}'
            })
            
            # Nettoyage mémoire périodique
            if batch_idx % 10 == 0 and device == "cuda":
                torch.cuda.empty_cache()
                gc.collect()
                
        except Exception as e:
            print(f"Erreur pendant l'entraînement (batch {batch_idx}): {e}")
            continue
    
    avg_loss = total_loss / max(num_batches, 1)
    print(f"Epoch {epoch+1}/{num_epochs} terminé - Perte moyenne: {avg_loss:.4f}")

print("Entraînement terminé !")

## Sauvegarder LoRA après l'entraînement

In [ ]:
# Sauvegarder dans le dossier LoRA du dataset
lora_save_path = lora_output / "lora_cadot_topdown"
lora_save_path.mkdir(exist_ok=True)

try:
    unet.save_attn_procs(str(lora_save_path))
    print(f"LoRA sauvegardé avec succès dans: {lora_save_path}")
    
    # Lister les fichiers sauvegardés
    print("Fichiers LoRA sauvegardés:")
    for file in lora_save_path.glob("*"):
        print(f"  {file.name}")
        
except Exception as e:
    print(f"Erreur lors de la sauvegarde: {e}")

## Test du modèle LoRA entraîné

In [ ]:
import matplotlib.pyplot as plt

# Recharger le pipeline avec LoRA
try:
    # Créer un nouveau pipeline pour le test
    if topdown_model_path.exists():
        test_pipe = StableDiffusionPipeline.from_single_file(
            str(topdown_model_path),
            torch_dtype=torch.float16 if device == "cuda" else torch.float32
        ).to(device)
    else:
        test_pipe = StableDiffusionPipeline.from_pretrained(
            "runwayml/stable-diffusion-v1-5",
            torch_dtype=torch.float16 if device == "cuda" else torch.float32
        ).to(device)
    
    # Charger les poids LoRA
    test_pipe.unet.load_attn_procs(str(lora_save_path))
    
    if device == "cuda":
        test_pipe.enable_model_cpu_offload()
    
    print("Pipeline de test chargé avec LoRA")

    # Prompts de test
    test_prompts = [
        "top-down aerial RGB image of an industrial area with several trucks and few cars",
        "top-down aerial RGB image of an urban area with many buildings",
        "top-down aerial RGB image of a residential area with houses and streets"
    ]

    # Générer des images de test
    fig, axes = plt.subplots(1, len(test_prompts), figsize=(15, 5))
    if len(test_prompts) == 1:
        axes = [axes]
    
    for i, prompt in enumerate(test_prompts):
        print(f"Génération: {prompt}")
        
        with torch.no_grad():
            image = test_pipe(
                prompt, 
                num_inference_steps=20,  # Réduire pour accélérer
                guidance_scale=7.5,
                height=512,
                width=512
            ).images[0]
        
        axes[i].imshow(image)
        axes[i].set_title(f"Test {i+1}")
        axes[i].axis('off')
        
        # Sauvegarder l'image
        output_path = lora_output / f"test_generation_{i+1}.png"
        image.save(output_path)
        print(f"Image sauvegardée: {output_path}")
    
    plt.tight_layout()
    plt.show()
    
    print("Test du modèle LoRA terminé avec succès !")

except Exception as e:
    print(f"Erreur lors du test: {e}")
    import traceback
    traceback.print_exc()

## Nettoyage mémoire

In [ ]:
# Libérer la mémoire GPU
if device == "cuda":
    torch.cuda.empty_cache()
    print("Cache GPU vidé")

import gc
gc.collect()
print("Nettoyage mémoire terminé")